# Fundamentals 00.2 - Bedrock Runtime API

Objetivo: comprobar `bedrock-runtime` mediante la misma fachada publica usada por otros providers. STS, control plane y llamadas raw pertenecen a un anexo operativo, no a esta demostracion canonica.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| RUN_BEDROCK_LIVE | 1 | Usa 0 para desactivar la llamada real. |
| BEDROCK_MODEL_ID | provider default | Override del modelo disponible en tu cuenta AWS. |
| AWS_REGION o AWS_DEFAULT_REGION | us-east-1 | Region de Bedrock. |
| credenciales AWS | cadena estandar | Perfil, variables o rol de instancia. |

## Contrato de la demostracion

El notebook prueba el provider Unicamente a traves de la fachada publica:

```text
toolkit.runtime - toolkit.system - system.agent - RunResult
```

La celda live se habilita con una variable explicita. Sin credenciales o endpoint, el notebook permanece ejecutable y muestra un skip estructurado. Cuando la variable esta activa, cualquier error real del provider debe ser visible.

In [ ]:
import os

import agentic_systems as toolkit

RUN_BEDROCK_LIVE = os.getenv("RUN_BEDROCK_LIVE", "1").strip().lower() in {"1", "true", "yes"}
REGION = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")
MODEL = os.getenv("BEDROCK_MODEL_ID")
AGENT_NAME = "bedrock_public_api_probe"

aws_environment = toolkit.aws_environment_snapshot()
aws_session = toolkit.boto3_session_snapshot(region_name=REGION)

toolkit.show_json({
    "package": toolkit.__name__,
    "version": toolkit.__version__,
    "run_live": RUN_BEDROCK_LIVE,
    "environment": aws_environment,
    "session": aws_session,
}, title="Preflight Bedrock")

## 1) Declarar runtime y limites

Modelo y region proceden del ambiente o de los defaults publicos. El notebook no fija un modelo o region como verdad universal.

In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=90,
    max_retries=1,
    max_tool_calls=1,
    max_turns=3,
    max_concurrency=1,
)

runtime = toolkit.runtime(
    provider="bedrock-runtime",
    model=MODEL,
    region=REGION,
    scheduler=scheduler,
    metadata={"tutorial": "bedrock-provider-api"},
)

toolkit.show_json(runtime.describe(), title="Bedrock RuntimeConfig")

## 2) Crear system, tool y agent

Ni el agente ni la tool conocen boto3. Cambiar de provider no cambia su contrato.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    """Verifica un simbolo contra la superficie publica instalada."""
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

system = toolkit.system(runtime=runtime)
agent = system.agent(
    name=AGENT_NAME,
    instructions=(
        "Usa inspect_public_api para verificar el simbolo solicitado. "
        "Responde con el nombre, si es publico y la version observada."
    ),
    tools=[inspect_public_api],
    contract=toolkit.AgentContract(
        must_call=["inspect_public_api"],
        completion="when_required_tools_satisfied",
    ),
    policy=toolkit.RunPolicy(
        max_turns=3,
        max_tool_calls=1,
        temperature=0.0,
        trace="compact",
        strict=True,
    ),
)

toolkit.show_json(agent.info(), title="Agente declarado")

La ejecucion live esta habilitada por defecto cuando la cadena AWS tiene credenciales. Usa RUN_BEDROCK_LIVE=0 para forzar un skip seguro; errores de permisos, region o modelo permanecen visibles.

In [ ]:
can_run = RUN_BEDROCK_LIVE and bool(aws_session.get("has_credentials"))

if can_run:
    result = agent.run(
        "Verifica si system pertenece a la API publica instalada.",
        mode="eval",
    )
    toolkit.human_result(result, title="Bedrock RunResult", show_lineage=True)
    toolkit.show_json(toolkit.run_result_output(result), title="Contrato normalizado")
else:
    result = None
    toolkit.show_json({
        "status": "skipped",
        "provider": "bedrock-runtime",
        "reason": "Configura credenciales AWS, o usa RUN_BEDROCK_LIVE=0.",
    }, title="Bedrock live gate")

## 4) API realmente ejercitada

Diagnostico AWS y ejecucion usan exclusivamente funciones publicas de `toolkit`.

In [ ]:
api_coverage = [
    "toolkit.aws_environment_snapshot",
    "toolkit.boto3_session_snapshot",
    "toolkit.scheduler",
    "toolkit.runtime",
    "toolkit.tool",
    "toolkit.system",
    "system.agent",
    "agent.run",
    "toolkit.human_result",
    "toolkit.run_result_output",
    "toolkit.show_json",
]

toolkit.show_json(api_coverage, title="Bedrock API coverage")

## Resultado esperado

Con live desactivado: snapshot seguro y configuracion observable. Con live activado: un `RunResult` real con engine `bedrock-runtime`; ningun secreto ni objeto boto3 aparece en la API del usuario.